# 01b geneer kwel input

In dit script wordt de kwel data ingeladen in in het juiste format weggeschreven.

In [1]:
import geopandas as gpd
import pandas as pd
import rasterio
from pathlib import Path
import xarray as xr
import rioxarray

#### input en output paden definieren.

In [2]:
refpoints_csv = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//#SCENARIO#_gebiedsindeling_RR_KNOPEN_tbv_Onderrand.csv"
raster_folder = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//seepage//"
output_csv = "..//..//WRIJ_RR_Unpaved_methode_02_input//rr_input_scenarios//scenarios//#SCENARIO#//kwel_per_RR_knoop.csv"

In [3]:
crs = "EPSG:28992"

#### Selecteer voor welke gebieden, scenario’s en periode de seepage moet worden gegenereerd.

De seepage krijgt een waarde per maand.

In [4]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [1, 2, 3]

scenarios = ["REF", "SCEN"]
# scenarios = ["REF"]

# LONG RUN
start_date = "2012-4-1"
end_date = "2018-12-31"

# path to the package containing the dummy-data
dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\"

In [5]:
date_range_data = pd.date_range(start_date, end_date, freq="MS")

#### Inlezen en bewerken van de seepage data

Maandelijkse rasterbestanden met seepage worden ingelezen en samegevoegd tot een tijdserie per scenario.

In [6]:
ds = xr.Dataset()
for scenario in scenarios:
    print(scenario)
    dir_seepage = Path(raster_folder.replace("#SCENARIO#", scenario))
    data_arrays = []
    for date in date_range_data:
        print(date)
        seepage_raster_filename = f"{scenario}_FLUX_L1L2_{date.strftime('%Y%m')}_MMD.ASC"
        if seepage_raster_filename == "REF_FLUX_L1L2_201512_MMD.ASC":
            data_arrays.append(da)
            continue
        if seepage_raster_filename == "REF_FLUX_L1L2_201707_MMD.ASC":
            data_arrays.append(da)
            continue
        da = rioxarray.open_rasterio(Path(dir_seepage, seepage_raster_filename), masked=True).squeeze("band")
        data_arrays.append(da)
    da = xr.concat(data_arrays, dim="time")
    ds[scenario] = da.assign_coords(time=date_range_data)

REF
2012-04-01 00:00:00
2012-05-01 00:00:00
2012-06-01 00:00:00
2012-07-01 00:00:00
2012-08-01 00:00:00
2012-09-01 00:00:00
2012-10-01 00:00:00
2012-11-01 00:00:00
2012-12-01 00:00:00
2013-01-01 00:00:00
2013-02-01 00:00:00
2013-03-01 00:00:00
2013-04-01 00:00:00
2013-05-01 00:00:00
2013-06-01 00:00:00
2013-07-01 00:00:00
2013-08-01 00:00:00
2013-09-01 00:00:00
2013-10-01 00:00:00
2013-11-01 00:00:00
2013-12-01 00:00:00
2014-01-01 00:00:00
2014-02-01 00:00:00
2014-03-01 00:00:00
2014-04-01 00:00:00
2014-05-01 00:00:00
2014-06-01 00:00:00
2014-07-01 00:00:00
2014-08-01 00:00:00
2014-09-01 00:00:00
2014-10-01 00:00:00
2014-11-01 00:00:00
2014-12-01 00:00:00
2015-01-01 00:00:00
2015-02-01 00:00:00
2015-03-01 00:00:00
2015-04-01 00:00:00
2015-05-01 00:00:00
2015-06-01 00:00:00
2015-07-01 00:00:00
2015-08-01 00:00:00
2015-09-01 00:00:00
2015-10-01 00:00:00
2015-11-01 00:00:00
2015-12-01 00:00:00
2016-01-01 00:00:00
2016-02-01 00:00:00
2016-03-01 00:00:00
2016-04-01 00:00:00
2016-05-01 00:00

??
Ik denk dat de seepage waardes per maand gekoppeld worden aan de RR knopen.

In [7]:
df_results = {}
for scenario in scenarios:
    print(scenario)
    refpoints_path = Path(refpoints_csv.replace("#SCENARIO#", scenario))
    refpoints = pd.read_csv(refpoints_path, sep=",")
    refpoints = refpoints.rename(columns={"xcoor": "x", "ycoor": "y"})

    ds_refpoints = refpoints.set_index(["x", "y"])["ID_RR_KNOOP"].to_xarray()
    ds[f"{scenario}_refpoints"] = ds_refpoints.reindex_like(ds)

    da_result = ds[scenario].where(ds[f"{scenario}_refpoints"].notnull()).groupby(ds[f"{scenario}_refpoints"]).mean(dim=("stacked_x_y"))
    df_result = da_result.to_dataframe()[scenario].reset_index().pivot(index="time", columns=f"{scenario}_refpoints", values=scenario)
    df_result.columns = ["sep_" + col for col in df_result.columns]
    df_results[scenario] = df_result
df_results = pd.concat(df_results, axis=1)

REF
SCEN


Voor 2012 is er nog geen seepage data beschikbaar. Daarom worden het gemiddelde per maand voor de periode 2012 - 2018 genomen.
Deze maandelijkse gemiddeldes worden ingevoegd voor de periode 2010 - 2012.

In [8]:
df_results_mean = df_results.groupby(df_results.index.month).mean()

In [9]:
df_start = pd.DataFrame(index=pd.date_range("2010-4-1", start_date, freq="MS"), columns=df_results.columns).iloc[:-1]

In [10]:
df_temp = df_results_mean.loc[df_start.index.month]
df_temp.index = df_start.index
df_results_total = pd.concat([df_temp, df_results])

#### wegschrijven per scenario

In [11]:
for scenario in scenarios:
    df_results_total[scenario].to_csv(output_csv.replace("#SCENARIO#", scenario))